# Optional: DB-Vergleich fehlerhaft vs. bereinigt

## Ziel
Dieses Notebook dokumentiert den Unterschied zwischen der fehlerhaften und der bereinigten Datenbankversion. Es dient der Nachvollziehbarkeit der Importkorrektur.

## Einordnung
Der CSV-Export bleibt als Diagnoseartefakt erhalten. Grafikdateien werden nicht exportiert; die Plots werden nur im Notebook angezeigt.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Sphinx-Dokumentationskontext

notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from config import ANOMALY_END, ANOMALY_START, DWH_DB_PATH, DWH_FAULTY_DB_PATH, PLOTS_DIR
from db_analysis import monthly_counts_from_db
from plotting_utils import apply_plot_style

apply_plot_style()


In [ ]:
# Adjust paths if needed
FAULTY_DB_PATH = DWH_FAULTY_DB_PATH
CLEAN_DB_PATH = DWH_DB_PATH

ARTIFACT_DIR = PLOTS_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not FAULTY_DB_PATH.exists():
    raise FileNotFoundError(f"Faulty DB missing: {FAULTY_DB_PATH.resolve()}")
if not CLEAN_DB_PATH.exists():
    raise FileNotFoundError(f"Clean DB missing: {CLEAN_DB_PATH.resolve()}")

print("DBs found:")
print("Fehlerhafte DB:", FAULTY_DB_PATH.resolve())
print("Bereinigte DB:", CLEAN_DB_PATH.resolve())
print("Artifacts in:", ARTIFACT_DIR.resolve())


In [ ]:
monthly = pd.concat(
    [
        monthly_counts_from_db(FAULTY_DB_PATH, "faulty"),
        monthly_counts_from_db(CLEAN_DB_PATH, "clean"),
    ],
    ignore_index=True,
 )

monthly = monthly.sort_values(["month", "source"]).reset_index(drop=True)
display(monthly.head(20))


In [ ]:
csv_path = ARTIFACT_DIR / "db_monthly_comparison_faulty_vs_clean.csv"
monthly.to_csv(csv_path, index=False)

pivot_newspapers = monthly.pivot(index="month_dt", columns="source", values="newspapers_count").sort_index()
pivot_context = monthly.pivot(index="month_dt", columns="source", values="context_count").sort_index()

# Plot-Stile: unterschiedliche Linien für fehlerhafte und bereinigte Datenbank
styles = {"faulty": "--", "clean": ":"}
marker_size = 6
line_alpha = 0.8

fig1, ax1 = plt.subplots(figsize=(12, 4.8))
for col in pivot_newspapers.columns:
    linestyle = styles.get(col, "-")
    ax1.plot(
        pivot_newspapers.index,
        pivot_newspapers[col],
        marker="o",
        markersize=marker_size,
        linewidth=2,
        linestyle=linestyle,
        alpha=line_alpha,
        label=col,
    )
ax1.axvspan(ANOMALY_START, ANOMALY_END, alpha=0.08, color="red")
ax1.set_title("Zeitungen pro Monat: fehlerhaft vs. bereinigt")
ax1.set_xlabel("Monat")
ax1.set_ylabel("Anzahl")
ax1.grid(alpha=0.3)
ax1.legend()
fig1.tight_layout()
fig1.patch.set_alpha(0)
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 4.8))
for col in pivot_context.columns:
    linestyle = styles.get(col, "-")
    ax2.plot(
        pivot_context.index,
        pivot_context[col],
        marker="o",
        markersize=marker_size,
        linewidth=2,
        linestyle=linestyle,
        alpha=line_alpha,
        label=col,
    )
ax2.axvspan(ANOMALY_START, ANOMALY_END, alpha=0.08, color="red")
ax2.set_title("Kontextzeilen pro Monat: fehlerhaft vs. bereinigt")
ax2.set_xlabel("Monat")
ax2.set_ylabel("Anzahl")
ax2.grid(alpha=0.3)
ax2.legend()
fig2.tight_layout()
fig2.patch.set_alpha(0)
plt.show()

print("CSV exportiert:")
print("-", csv_path.resolve())


In [ ]:
feb = monthly[monthly['month'].str.endswith('-02')].copy()
display(feb.sort_values(['month', 'source']).reset_index(drop=True))
